Groundwater | Transport Modeling Track

# Step 5: Calibration — and from here, your project

> **The 10 steps at a glance:** 1-Goal → 2-Perceptual → 3-MODFLOW → 4-Build → **5-Calibrate** → 6-Validate → 7-Sensitivity → 8-Apply → 9-Document → 10-Communicate

**Story so far:** In [04t_model_implementation.ipynb](04t_model_implementation.ipynb) you built the coupled flow-and-transport doublet, confronted the grid–Péclet problem, and locked the conservative-solute transport parameters ($\alpha_L = 10$ m, $\alpha_T = 1$ m, $n_e = 0.20$).

**Why this notebook is short.** A *full* transport calibration — estimating $\alpha_L$, $\alpha_T$ and $n_e$ from observed plume data — is **beyond the scope of this course**: in the field you rarely have enough concentration observations to constrain those parameters, and they trade off against one another. The **calibration rubric is carried entirely by the flow phase** of your project (heads, fluxes, PEST++ in the flow track). So Step 5 is deliberately a **short bridge**: it hands you off from the taught notebooks into the **Weeks 11–12 transport project** and the written report.

**Forward:** → [08t_model_application.ipynb](08t_model_application.ipynb) (the keystone doublet study) → your **written report & presentation** (Steps 9–10 — there are no 06t/07t notebooks in the transport track).

| | |
|---|---|
| **Read time** | ~10–15 min |
| **Optional α_L demo** | +5 min (light analytical calc) |

## What this bridge gives you

This notebook does **not** ask you to calibrate anything. Instead it:

1. lays out the **deliverable checklist** for the transport phase of your report;
2. restates the **defensible-threshold rule** (which claims you may make, and which you may not);
3. offers **one optional, illustrative** $\alpha_L$ demonstration so you can predict-then-see how dispersivity reshapes a breakthrough curve;
4. closes with a short self-check.

> **Conservative-solute framing.** Throughout the transport track we model a **conservative tracer / contaminant** (no sorption, no decay). The transport parameters are **locked**: $\alpha_L = 10$ m, $\alpha_T = 1$ m, $n_e = 0.20$. In your project you do **not** re-calibrate them — you *lock* them and instead explore how sensitive your answer is to $\alpha_L$.

## Deliverable — what the transport phase of your report must contain

The transport study is the Weeks 11–12 continuation of your group's flow scenario on the same Limmat model. Your report's **transport section** must contain:

- [ ] **A framed transport question.** State precisely what you ask of the model — e.g. *"Does the reinjected tracer reach the abstraction well above concentration X, and when?"* Frame it **longitudinally / at a receptor**, not as a contaminated-area map (see the warning below).
- [ ] **A breakthrough / recirculation result.** The breakthrough curve at the receptor (or the steady-state recirculation fraction $C_\infty/c_\text{inj}$ for the doublet), read off your model — the quantitative answer to your question.
- [ ] **An $\alpha_L$ sensitivity statement.** Re-run with a different $\alpha_L$ (e.g. baseline 10 m → 5 m) and report how arrival time, peak and tail move. Sensitivity *replaces* calibration here.
- [ ] **Your model files.** The modified MODFLOW 6 GWT model (inputs + outputs), so your result is reproducible.
- [ ] **The defensible-threshold judgment.** An explicit statement of which of your threshold claims are defensible and which are not — see below. **This is graded.**

> **Where this is assessed.** The defensible-threshold judgment maps onto the report rubric under **Results & Interpretation (20%)** — the *limitations* discussion — and **Conceptual Model (15%)** — the *assumptions* you state and defend. It is the model-quality judgment node from the course's quality-judgment track.

## ⚠️ Defensible-threshold rule (taught, not fixed)

You met this in [04t](04t_model_implementation.ipynb) and apply it in [08t](08t_model_application.ipynb). It is the single most important judgment in your transport report:

| You **may** claim (grid-robust) | You may **not** claim (numerical artefact) |
|:--|:--|
| whether and roughly **how much** solute reaches the receptor (recirculation fraction) | the **lateral width** of the contaminated zone |
| **receptor arrival time** / breakthrough at a well | the **exact position** of a concentration contour |
| the **longitudinal (centreline) reach** of the plume | a **mapped contaminated area** |

**Why.** The transverse grid Péclet stays $Pe_T \approx 10$–$12 \gg 2$ even on the refined corridor: the lateral plume width is set by **numerical dispersion (cell size)**, not by physics, and **no feasible grid** can fix it. So restrict every threshold claim to the **flow axis / receptors**, and explicitly **warn against lateral contaminated-area or exact-contour statements** in your report.

## Self-check

Before you carry these parameters into your project, make sure you understand *why* transport calibration is being handled this way.

In [ ]:
import sys, os
sys.path.insert(0, '../../_SUPPORT/src/scripts/scripts_exercises')
from shared_functions import create_multiple_choice

create_multiple_choice('task_t05_checkpoint_1')

## Optional — predict, then run: how does $\alpha_L$ reshape a breakthrough? *(illustrative)*

This is a **light analytical calculation** (no MODFLOW run — it finishes in well under a second), included so you can build intuition before you run the *real* $\alpha_L$ sensitivity on the doublet in your project.

**Predict first.** For an **instantaneous pulse** released upstream and observed at a fixed distance, how does *increasing* $\alpha_L$ change the breakthrough curve — earlier or later first arrival? higher or lower peak?

We use the 1D advection–dispersion **pulse** solution `tracer_test_utils.analytical_btc`, evaluated for three dispersivities ($\alpha_L = 5, 10, 20$ m) at a receptor $x = 100$ m downstream in idealised uniform flow ($v = 1$ m/d, $n_e = 0.20$).

> **Units note — flux-averaged concentration (the `/v` handling).** As written, `analytical_btc` returns the pulse solution **without the $1/v$ prefactor**, so its raw output carries units of $\text{g}\,\text{m}^{-2}\,\text{d}^{-1}$, not a concentration. We therefore **divide the result by the seepage velocity $v$** in the cell below, which yields the genuine **flux-averaged** concentration in $\text{mg/L}$ ($=\text{g/m}^3$). We do the $/v$ *here* rather than editing the shared function: its only other caller fits breakthrough-curve *shape* in the legacy tracer-test utilities and is unaffected by a constant factor, so silently changing the function would need a coordinated re-check of that caller.

> **This curve is illustrative / conservative — do NOT compare it to a regulatory threshold.** It assumes idealised uniform 1D flow and an arbitrary injected mass; only its *shape* trend carries the lesson.

In [ ]:
# --- Optional alpha_L demonstration (light analytical calc; safe to leave on) ---
RUN_ALPHA_DEMO = True   # set False to skip; this is a sub-second analytical calc, not a model run

import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, '../../_SUPPORT/src')
from tracer_test_utils import analytical_btc

if RUN_ALPHA_DEMO:
    # Idealised 1D uniform-flow pulse (illustrative scenario)
    v   = 1.0       # seepage velocity [m/d]
    x   = 100.0     # receptor distance [m]
    n_e = 0.20      # locked effective porosity [-]
    M   = 1000.0    # injected mass [g] (arbitrary -> sets only the y-scale)
    A   = 10.0      # cross-sectional flow area [m^2]
    D_m = 8.64e-5   # molecular diffusion [m^2/d] (negligible here)
    t   = np.linspace(0.1, 400.0, 4000)   # time since injection [d]

    fig, ax = plt.subplots(figsize=(9, 5))
    for aL in [5.0, 10.0, 20.0]:
        D_L    = aL * v + D_m                              # longitudinal dispersion coeff [m^2/d]
        c_raw  = analytical_btc(t, x, v, D_L, M, A, n_e)   # g/(m^2 d) -- missing the 1/v prefactor
        c_flux = c_raw / v                                 # /v -> flux-averaged mg/L (= g/m^3)
        pk = c_flux.max(); t_pk = t[c_flux.argmax()]
        ax.plot(t, c_flux, lw=2,
                label=f'alpha_L = {aL:.0f} m  (peak {pk:.2f} mg/L at {t_pk:.0f} d)')

    ax.axvline(x / v, color='0.6', ls='--', lw=1)
    ax.text(x / v, ax.get_ylim()[1] * 0.96, ' advective arrival x/v',
            color='0.4', va='top', fontsize=9)
    ax.set_xlabel('time since injection [d]')
    ax.set_ylabel('flux-averaged concentration [mg/L]')
    ax.set_title('Illustrative pulse breakthrough at x = 100 m -- effect of alpha_L\n'
                 '(conservative tracer; do NOT compare to a regulatory threshold)')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print('alpha_L demo skipped (RUN_ALPHA_DEMO = False).')

**What the demo shows ($\alpha_L$ key for a pulse).** Increasing $\alpha_L$ spreads the pulse more, so the breakthrough arrives **earlier** (earlier first arrival / leading toe), peaks **lower and broader**, and develops a **longer tail**. Mass is conserved, so a wider curve must be flatter — and the centre-of-mass arrival stays near the advective time $x/v$ (dashed line). This is exactly the pulse behaviour you will reproduce — on the real doublet — when you run your project's $\alpha_L$ sensitivity.

> *(For a **continuous** source the key differs: larger $\alpha_L$ gives an earlier first-arrival shoulder and a less-steep front, but the 50% breakthrough stays near $x/v$, so full plateau is reached no sooner.)*

## What you take forward

| Into your project | Value |
|---|---|
| $\alpha_L$ | 10 m (locked baseline; sweep to 5 m for sensitivity) |
| $\alpha_T$ | 1 m (locked; the transverse axis is numerically under-resolved) |
| $n_e$ | 0.20 (locked) |
| Calibration | carried by the **flow** phase — not repeated for transport |
| Rigour source | **$\alpha_L$ sensitivity** + the **defensible-threshold judgment**, not parameter tuning |

You now have everything you need to start the transport phase of your project:
- run the keystone study in **[08t_model_application.ipynb](08t_model_application.ipynb)**, then
- write up the deliverable checklist above in your **report** (Steps 9–10).

**Navigation:** ← [04t_model_implementation.ipynb](04t_model_implementation.ipynb)  ·  → [08t_model_application.ipynb](08t_model_application.ipynb)  ·  → your **written report & presentation**

> *Section-completion note:* this notebook intentionally does **not** call `create_section_completion_marker` (that tracker is bound to 02t).

## References

\[1\] Domenico, P.A. & Schwartz, F.W. (1998). *Physical and Chemical Hydrogeology* (2nd ed.). Wiley. — analytical transport solutions (course primary text).

\[2\] Kreft, A. & Zuber, A. (1978). On the physical meaning of the dispersion equation and its solutions for different initial and boundary conditions. *Chemical Engineering Science*, 33(11), 1471–1480. — flux- vs resident-averaged concentrations.

\[3\] Langevin, C.D., et al. (2022). MODFLOW 6 Description of Input and Output. U.S. Geological Survey Techniques and Methods 6-A62. — GWT transport packages.